# Análise de Dados com Python [T1]
## Mini-Projeto Avaliativo - Módulo 1 - Semana 07
### Rodrigo Vieira

## Obejtivo
#### realizar uma Análise Exploratória da base Varejo seguindo etapas claras, documentadas e reproduzíveis.

In [1]:
# Imports das Bibliotecas Necessárias

import pandas as pd
import kagglehub
import shutil
from pathlib import Path

In [2]:
# Download e carga da base

# Define o caminho da pasta de dados
data_dir = Path('data')
data_dir.mkdir(exist_ok=True)

# Baixa o dataset do Kaggle
path = kagglehub.dataset_download('namespaiva/base-varejo')

# Procura pelo arquivo CSV dentro da pasta baixada
# Prioriza 'Varejo.csv' se existir, caso contrário pega o primeiro CSV encontrado
path_obj = Path(path)
csv_files = list(path_obj.rglob('*.csv'))

if not csv_files:
    raise FileNotFoundError('Nenhum arquivo CSV encontrado no dataset.')

# Seleciona o arquivo alvo
target_file = next((f for f in csv_files if f.name == 'Varejo.csv'), csv_files[0])

# Define o caminho final e copia o arquivo
final_path = data_dir / 'Varejo.csv'
shutil.copy2(target_file, final_path)

# Carrega os dados em um DataFrame
df = pd.read_csv(final_path, sep=';', encoding='utf-8')

# Exibe as primeiras linhas
df.head()

,DATA,CO_ID,CL_ID,CL_GENERO,CL_EC,CL_FHL,CL_SEG,PR_ID,PR_CAT,PR_NOME,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13
0,01/02/2019,1000,534,M,4,1,C,67,BEBIDAS,REFRIGERANTE GUARANA,NaN,NaN,NaN,NaN
1,01/02/2019,1000,534,M,4,1,C,70,BEBIDAS,REFRIGERANTE OUTROS,NaN,NaN,NaN,NaN
2,01/02/2019,1000,534,M,4,1,C,178,HIGIENE,LENCO UMEDECIDO,NaN,NaN,NaN,NaN
3,01/02/2019,1000,534,M,4,1,C,4,ALIMENTOS,ABACAXI,NaN,NaN,NaN,NaN
4,01/02/2019,1000,534,M,4,1,C,175,LIMPEZA,LIMPADOR MULTIUSO,NaN,NaN,NaN,NaN


In [3]:
# Drop de colunas não nomeadas 'Unnamed' (verificando o dataset no kaggle, essas colunas não existem)
# Em uma checagem prévia essas colunas aparecem com todos os registros ausentes.
df = df.loc[:, ~df.columns.str.startswith('Unnamed:')]

### Problema 1
O arquivo baixado garrega um Dataset com 4 colunas inexistentes. 

Após verificar que todas as linhas dessas colunas representam valores ausentes, eu decidi eliminar essas colunas

In [4]:
# Verifica a estrutura básica do df.
df.shape

(830000, 10)

In [5]:
# exibe os tipo de dados
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 830000 entries, 0 to 829999
Data columns (total 10 columns):
 #   Column     Non-Null Count   Dtype
---  ------     --------------   -----
 0   DATA       830000 non-null  str  
 1   CO_ID      830000 non-null  int64
 2   CL_ID      830000 non-null  int64
 3   CL_GENERO  830000 non-null  str  
 4   CL_EC      830000 non-null  int64
 5   CL_FHL     830000 non-null  int64
 6   CL_SEG     830000 non-null  str  
 7   PR_ID      830000 non-null  int64
 8   PR_CAT     830000 non-null  str  
 9   PR_NOME    830000 non-null  str  
dtypes: int64(5), str(5)
memory usage: 63.3 MB


In [6]:
# Contagem de valores ausentes
# Aparentemente não há valores ausentes
df.isna().sum().sort_values(ascending=False)

DATA         0
CO_ID        0
CL_ID        0
CL_GENERO    0
CL_EC        0
CL_FHL       0
CL_SEG       0
PR_ID        0
PR_CAT       0
PR_NOME      0
dtype: int64

In [7]:
# Verificando valores únicos na coluna PR_CAT
# O valor '#N/D' chama a atenção. Será um valor ausente??
df['PR_CAT'].value_counts()

PR_CAT
ALIMENTOS     434767
HIGIENE       155574
LIMPEZA       145754
BEBIDAS        43299
PET            32399
ACESSORIOS     14557
#N/D            3650
Name: count, dtype: int64

In [8]:
# Verificando valores únicos na coluna PR_NOME
# O valor '#N/D' chama a atenção. Será um valor ausente??
# É a mesma quantidade de linhas da coluna PR_CAT (3650)
df['PR_NOME'].value_counts()

PR_NOME
PRESUNTO COZIDO          14381
SARDINHA                  7490
GEL                       7399
BANANA                    7385
DESENGORDURANTE           7378
                         ...  
#N/D                      3650
ABSORVENTE                3639
ALCOOL                    3611
ALIMENTO PARA PASSARO     3605
ALHO                      3574
Name: count, Length: 118, dtype: int64

In [9]:
# verificando as demais colunas onde os valores de PR_CAT e PR_NOME são iguais a '#N/D'
# O Valos '#N/D' possui um PR_ID único (107), o que indica que é um valor válido. 
df[df['PR_CAT'] == '#N/D'].head()

,DATA,CO_ID,CL_ID,CL_GENERO,CL_EC,CL_FHL,CL_SEG,PR_ID,PR_CAT,PR_NOME
82,01/02/2019,1078,290,F,1,0,B,107,#N/D,#N/D
223,01/02/2019,1103,957,M,2,3,B,107,#N/D,#N/D
640,01/02/2019,1482,453,F,1,0,B,107,#N/D,#N/D
857,01/02/2019,1683,438,F,1,0,B,107,#N/D,#N/D
917,01/02/2019,1793,608,F,3,0,C,107,#N/D,#N/D


### Problema 2

Após analisar a base, o valor #N/D nas colunas PR_CAT e PR_NOME chamou atenção por aparecer 3.650 vezes nas duas colunas. Primeiro, considerei que pudesse ser um valor ausente, mas ao olhar melhor as outras informações dessas linhas, vi que ele está ligado a um PR_ID específico (107), o que indica que esse registro existe de fato na base. 

Mesmo assim, como esse grupo representa uma parte pequena dos dados e não contribui para a análise das categorias de produtos, optei por remover as linhas em que PR_CAT e PR_NOME são iguais a #N/D para deixar a análise mais limpa e objetiva.

In [10]:
# Aliminando linhas onde 'PR_CAT' e 'PR_NOME' = '#N/D'
df = df[~((df['PR_CAT'] == '#N/D') & (df['PR_NOME'] == '#N/D'))].copy()

In [11]:
# Nova verificação da estrutura básica do df.
print(df.shape)
df.info()

(826350, 10)
<class 'pandas.DataFrame'>
Index: 826350 entries, 0 to 829999
Data columns (total 10 columns):
 #   Column     Non-Null Count   Dtype
---  ------     --------------   -----
 0   DATA       826350 non-null  str  
 1   CO_ID      826350 non-null  int64
 2   CL_ID      826350 non-null  int64
 3   CL_GENERO  826350 non-null  str  
 4   CL_EC      826350 non-null  int64
 5   CL_FHL     826350 non-null  int64
 6   CL_SEG     826350 non-null  str  
 7   PR_ID      826350 non-null  int64
 8   PR_CAT     826350 non-null  str  
 9   PR_NOME    826350 non-null  str  
dtypes: int64(5), str(5)
memory usage: 69.4 MB


In [12]:
# Nova contagem de valores ausentes
# Verificar se não criamos valores ausntes inadivertidamente
df.isna().sum().sort_values(ascending=False)

DATA         0
CO_ID        0
CL_ID        0
CL_GENERO    0
CL_EC        0
CL_FHL       0
CL_SEG       0
PR_ID        0
PR_CAT       0
PR_NOME      0
dtype: int64

In [13]:
# Exibe as primeiras linhas
df.head()

,DATA,CO_ID,CL_ID,CL_GENERO,CL_EC,CL_FHL,CL_SEG,PR_ID,PR_CAT,PR_NOME
0,01/02/2019,1000,534,M,4,1,C,67,BEBIDAS,REFRIGERANTE GUARANA
1,01/02/2019,1000,534,M,4,1,C,70,BEBIDAS,REFRIGERANTE OUTROS
2,01/02/2019,1000,534,M,4,1,C,178,HIGIENE,LENCO UMEDECIDO
3,01/02/2019,1000,534,M,4,1,C,4,ALIMENTOS,ABACAXI
4,01/02/2019,1000,534,M,4,1,C,175,LIMPEZA,LIMPADOR MULTIUSO


### Ajustando os Tipos de Dados

Na própria página do Kaggle, no arquivo 'Projeto III - Anlise Exploratria de Dados Utilizando o Python ou RStudio.pdf', encontrei essa descrição que condiz com um dicionário de dados para o Dataset Varejo.

- DATA: Data da compra;
- CO_ID: Identificação do número de compra (número da nota fiscal);
- CL_ID: Identificação do cliente (número do cliente);
- CL_GENERO: Sexo biológico informado pelo cliente; 
- CL_EC: Estado civil do cliente:
    - 1: Casado ou união estával;
    - 2: Divorciado;
    - 3: Separado;
    - 4: Solteiro;
    - 5: Viúvo.
- CL_FHL: Número de filhos do cliente;
- CL_SEG: Segmentação econômica do cliente (classe A, B ou C);
- PR_ID: Código do produto (SKU) adquirido;
- PR_CAT: Categoria do produto adquirido;
- PR_NOME: Nome do produto adquirido.

Vou ajustar os tipos de dados a partir desse dicionário.

In [14]:
# Ajustando os tipos de dados conforme o dicionário encontrado.

# Ajuste da coluna de data
df['DATA'] = pd.to_datetime(df['DATA'], errors='coerce')

# Conversão de IDs para string
for col in ['CO_ID', 'CL_ID', 'PR_ID']:
    df[col] = df[col].astype('string')

# Mapeamento e conversão de estado civil
mapa_ec = {1: 'Casado/União estável', 2: 'Divorciado', 3: 'Separado', 4: 'Solteiro', 5: 'Viúvo'}
df['CL_EC'] = df['CL_EC'].map(mapa_ec).astype('category')

# Otimização de número de filhos
df['CL_FHL'] = pd.to_numeric(df['CL_FHL'], downcast='integer')

# Conversão de colunas categóricas de texto
cat_cols = ['CL_GENERO', 'CL_SEG', 'PR_CAT', 'PR_NOME']
for col in cat_cols:
    df[col] = df[col].astype(str).str.strip().astype('category')

In [15]:
# Nova verificação sobre valores ausentes
df.isna().sum().sort_values(ascending=False)

DATA         482566
CO_ID             0
CL_ID             0
CL_GENERO         0
CL_EC             0
CL_FHL            0
CL_SEG            0
PR_ID             0
PR_CAT            0
PR_NOME           0
dtype: int64

In [16]:
# Exibição dos resultados
print(df.shape)
df.info()

(826350, 10)
<class 'pandas.DataFrame'>
Index: 826350 entries, 0 to 829999
Data columns (total 10 columns):
 #   Column     Non-Null Count   Dtype         
---  ------     --------------   -----         
 0   DATA       343784 non-null  datetime64[us]
 1   CO_ID      826350 non-null  string        
 2   CL_ID      826350 non-null  string        
 3   CL_GENERO  826350 non-null  category      
 4   CL_EC      826350 non-null  category      
 5   CL_FHL     826350 non-null  int8          
 6   CL_SEG     826350 non-null  category      
 7   PR_ID      826350 non-null  string        
 8   PR_CAT     826350 non-null  category      
 9   PR_NOME    826350 non-null  category      
dtypes: category(5), datetime64[us](1), int8(1), string(3)
memory usage: 36.3 MB


In [17]:
# Exibe as primeiras linhas pós ajustes
df.head()

,DATA,CO_ID,CL_ID,CL_GENERO,CL_EC,CL_FHL,CL_SEG,PR_ID,PR_CAT,PR_NOME
0,2019-01-02,1000,534,M,Solteiro,1,C,67,BEBIDAS,REFRIGERANTE GUARANA
1,2019-01-02,1000,534,M,Solteiro,1,C,70,BEBIDAS,REFRIGERANTE OUTROS
2,2019-01-02,1000,534,M,Solteiro,1,C,178,HIGIENE,LENCO UMEDECIDO
3,2019-01-02,1000,534,M,Solteiro,1,C,4,ALIMENTOS,ABACAXI
4,2019-01-02,1000,534,M,Solteiro,1,C,175,LIMPEZA,LIMPADOR MULTIUSO
